In [1]:
import torch
from torch import nn
import numpy as np
import gym
import torch.nn.functional as F

setattr(np, "bool8", np.bool)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class Actor(nn.Module):
    def __init__(self, obs_n=4, act_n=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_n, 128),
            nn.ReLU(),
            nn.Linear(128, act_n)
        )
    
    def forward(self, obs):
        return self.net(obs)

class Critor(nn.Module):
    def __init__(self, obs_n=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_n, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, obs):
        return self.net(obs)

In [ ]:
class PPO:
    def __init__(self, obs_n=4, act_n=2, actor_lr=1e-3, critor_lr=1e-2,gamma=0.98, device="cpu"):
        self.device = device
        self.actor = Actor(obs_n, act_n).to(self.device)
        self.critor = Critor(obs_n).to(self.device)
        self.optimizer_actor = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.optimizer_critor = torch.optim.Adam(self.critor.parameters(), lr=critor_lr)
        self.gamma = gamma
        self.lmdb = 0.98
        self.epoch = 5
        self.eps = 0.2
        
    
    def take_action(self, obs):
        obs = torch.tensor([obs], dtype=torch.float).to(self.device)
        logits = self.actor(obs)
        probs = torch.distributions.Categorical(logits=logits)
        action = probs.sample()
        return action.item()
    
    def compute_advantages(self, td_delta, gamma, lmdb):
        advantages = []
        advantage = 0.0
        td_delta = td_delta.detach().cpu().numpy()
        for delta in td_delta[::-1]:
            advantage = advantage * gamma * lmdb + delta
            advantages.append(advantage)
        
        advantages = advantages[::-1]
        return torch.tensor(advantages).to(self.device)
    
    
    def learn(self, obs_list, action_list, reward_list, next_obs_list, done_list):
        obs = torch.tensor(obs_list, dtype=torch.float).to(self.device)
        action = torch.tensor(action_list, dtype=torch.long).to(self.device)
        reward = torch.tensor(reward_list, dtype=torch.float).to(self.device).reshape(-1, 1)
        next_obs = torch.tensor(next_obs_list, dtype=torch.float).to(self.device)
        done = torch.tensor(done_list, dtype=torch.float).to(self.device).reshape(-1, 1)


        td_target = reward + self.gamma * self.critor(next_obs) * (1 - done)
        td_delta = td_target - self.critor(obs)


        advantages = self.compute_advantages(td_delta, self.gamma, self.lmdb)
        
        old_logit = self.actor(obs)
        old_log_prob = -F.cross_entropy(old_logit, action).detach()

        for _ in range(self.epoch):
            logit = self.actor(obs)
            log_prob = -F.cross_entropy(logit, action)

            ratio = torch.exp(log_prob - old_log_prob)

            sup1 = ratio * advantages
            sup2 = torch.clip(ratio, 1-self.eps, 1+self.eps)
            actor_loss = torch.mean(-torch.min(sup1, sup2))


            critor_loss = F.mse_loss(td_target.detach(), self.critor(obs))

            self.optimizer_actor.zero_grad()
            actor_loss.backward()
            self.optimizer_actor.step()

            self.optimizer_critor.zero_grad()
            critor_loss.backward()
            self.optimizer_critor.step()



In [4]:
env = gym.make("CartPole-v0")

episode = 200

actor_lr = 1e-3
critor_lr = 1e-2
gamma = 0.98
device = "cpu"
agent = PPO(
    obs_n=4,
    act_n=2,
    actor_lr=actor_lr,
    critor_lr=critor_lr,
    gamma=gamma,
    device=device,
)

for i in range(episode):
    obs_list , action_list, reward_list, next_obs_list, done_list = [], [], [], [], []
    obs = env.reset()[0]
    while True:
        action = agent.take_action(obs)
        next_observation, reward, done, truncated, info = env.step(action)

        obs_list.append(obs)
        action_list.append(action)
        reward_list.append(reward)
        next_obs_list.append(next_observation)
        done_list.append(done)

        obs = next_observation
        if done:
            break
    
    agent.learn(obs_list , action_list, reward_list, next_obs_list, done_list)
    print(f"Episode: {i+1}, Reward:{sum(reward_list)}")
    

d:\software\miniconda3\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
C:\Users\z1573\AppData\Local\Temp\ipykernel_4512\1572982313.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  obs = torch.tensor([obs], dtype=torch.float).to(self.device)


Episode: 1, Reward:24.0
Episode: 2, Reward:15.0
Episode: 3, Reward:22.0
Episode: 4, Reward:16.0
Episode: 5, Reward:33.0
Episode: 6, Reward:39.0
Episode: 7, Reward:38.0
Episode: 8, Reward:23.0
Episode: 9, Reward:13.0
Episode: 10, Reward:18.0
Episode: 11, Reward:61.0
Episode: 12, Reward:14.0
Episode: 13, Reward:16.0
Episode: 14, Reward:60.0
Episode: 15, Reward:22.0
Episode: 16, Reward:25.0
Episode: 17, Reward:25.0
Episode: 18, Reward:11.0
Episode: 19, Reward:12.0
Episode: 20, Reward:30.0
Episode: 21, Reward:13.0
Episode: 22, Reward:15.0
Episode: 23, Reward:14.0
Episode: 24, Reward:15.0
Episode: 25, Reward:10.0
Episode: 26, Reward:10.0
Episode: 27, Reward:10.0
Episode: 28, Reward:10.0
Episode: 29, Reward:15.0
Episode: 30, Reward:10.0
Episode: 31, Reward:14.0
Episode: 32, Reward:22.0
Episode: 33, Reward:11.0
Episode: 34, Reward:10.0
Episode: 35, Reward:11.0
Episode: 36, Reward:10.0
Episode: 37, Reward:9.0
Episode: 38, Reward:12.0
Episode: 39, Reward:14.0
Episode: 40, Reward:17.0
Episode: 4